# ConvBERT



In [ ]:
%pip install -q \
  torch==2.0.1 \
  transformers==4.33.1 \
  datasets==2.14.5 \
  accelerate==0.23.0 \
  evaluate==0.4.0 \
  requests==2.31.0 \
  numpy pandas tqdm sentencepiece


In [ ]:
import json, random, re, string, time
from pathlib import Path
from collections import Counter
import numpy as np
import torch
from datasets import Dataset, DatasetDict

SEED=42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DATA_REPO="https://github.com/Gokcimen/Home_Appliance_Dataset"
!rm -rf /content/Home_Appliance_Dataset
!git clone -q --depth 1 {DATA_REPO}.git /content/Home_Appliance_Dataset

DATA_ROOT=Path("/content/Home_Appliance_Dataset")

def flatten(path):
    raw=json.loads(Path(path).read_text(encoding="utf-8"))
    rows=[]
    for article in raw["data"]:
        title=article["title"]
        for para in article["paragraphs"]:
            context=para["context"]
            for qa in para["qas"]:
                rows.append({
                    "id":str(qa["id"]),
                    "title":title,
                    "context":context,
                    "question":qa["question"],
                    "answers":{
                        "text":[a["text"] for a in qa["answers"]],
                        "answer_start":[int(a["answer_start"]) for a in qa["answers"]],
                    }
                })
    return rows

raw_datasets=DatasetDict({
    "train":Dataset.from_list(flatten(DATA_ROOT/"train.json")),
    "validation":Dataset.from_list(flatten(DATA_ROOT/"dev.json")),
    "test":Dataset.from_list(flatten(DATA_ROOT/"test.json")),
})

assert len(raw_datasets["train"])==8000
assert len(raw_datasets["validation"])==1000
assert len(raw_datasets["test"])==1000

ids={s:set(raw_datasets[s]["id"]) for s in raw_datasets}
assert not ids["train"]&ids["validation"]
assert not ids["train"]&ids["test"]
assert not ids["validation"]&ids["test"]
all_ids=set().union(*ids.values())
assert len(all_ids)==10000
assert {int(x) for x in all_ids}==set(range(1,10001))

titles={s:set(raw_datasets[s]["title"]) for s in raw_datasets}
assert not titles["train"]&titles["validation"]
assert not titles["train"]&titles["test"]
assert not titles["validation"]&titles["test"]
assert len(set().union(*titles.values()))==1111

print("train",len(raw_datasets["train"]))
print("validation",len(raw_datasets["validation"]))
print("test",len(raw_datasets["test"]))
print("products",len(set().union(*titles.values())))


In [ ]:
MODEL_KEY="convbert"
MODEL_ID="YituTech/conv-bert-base"
print(MODEL_ID)

In [ ]:
from transformers import AutoConfig
config=AutoConfig.from_pretrained(MODEL_ID)
print(config)


In [ ]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,use_fast=True)
model=AutoModelForQuestionAnswering.from_pretrained(MODEL_ID)

MAX_LENGTH=384
STRIDE=128

def preprocess(examples):
    questions=[q.strip() for q in examples["question"]]
    tokenized=tokenizer(
        questions,
        examples["context"],
        max_length=MAX_LENGTH,
        truncation="only_second",
        stride=STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )
    offsets=tokenized.pop("offset_mapping")
    sample_map=tokenized.pop("overflow_to_sample_mapping")
    starts=[]
    ends=[]

    for i,offset in enumerate(offsets):
        sample_idx=sample_map[i]
        answer=examples["answers"][sample_idx]
        start_char=answer["answer_start"][0]
        end_char=start_char+len(answer["text"][0])
        seq=tokenized.sequence_ids(i)

        context_indexes=[j for j,v in enumerate(seq) if v==1]
        c0=context_indexes[0]
        c1=context_indexes[-1]

        if offset[c0][0]>start_char or offset[c1][1]<end_char:
            starts.append(0)
            ends.append(0)
            continue

        j=c0
        while j<=c1 and offset[j][0]<=start_char:
            j+=1
        starts.append(j-1)

        j=c1
        while j>=c0 and offset[j][1]>=end_char:
            j-=1
        ends.append(j+1)

    tokenized["start_positions"]=starts
    tokenized["end_positions"]=ends
    return tokenized

train_tokenized=raw_datasets["train"].map(
    preprocess,batched=True,remove_columns=raw_datasets["train"].column_names
)
validation_tokenized=raw_datasets["validation"].map(
    preprocess,batched=True,remove_columns=raw_datasets["validation"].column_names
)


In [ ]:
from transformers import TrainingArguments, Trainer, DefaultDataCollator

OUTPUT_DIR=f"/content/outputs/{MODEL_KEY}"

training_args=TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    warmup_ratio=0.10,
    max_grad_norm=1.0,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    save_total_limit=3,
    seed=42,
    data_seed=42,
    report_to="none",
)

trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=validation_tokenized,
    data_collator=DefaultDataCollator(),
    processing_class=tokenizer,
)

train_result=trainer.train()
trainer.save_model(f"{OUTPUT_DIR}/final")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final")


In [ ]:
def normalize_answer(text):
    text=str(text).lower()
    text="".join(c for c in text if c not in string.punctuation)
    text=re.sub(r"\b(a|an|the)\b"," ",text)
    return " ".join(text.split())

def exact_match(pred,gold):
    return float(normalize_answer(pred)==normalize_answer(gold))

def token_f1(pred,gold):
    p=normalize_answer(pred).split()
    g=normalize_answer(gold).split()
    if not p or not g:
        return float(p==g)
    same=sum((Counter(p)&Counter(g)).values())
    if same==0:
        return 0.0
    precision=same/len(p)
    recall=same/len(g)
    return 2*precision*recall/(precision+recall)

def score_rows(rows):
    return {
        "n":len(rows),
        "EM":100*sum(exact_match(r["prediction"],r["gold"]) for r in rows)/len(rows),
        "F1":100*sum(token_f1(r["prediction"],r["gold"]) for r in rows)/len(rows),
        "mean_latency_seconds":sum(float(r.get("latency_seconds",0)) for r in rows)/len(rows),
    }


In [ ]:
from transformers import pipeline
import glob, os, json, time
import pandas as pd

def evaluate_checkpoint(checkpoint):
    qa=pipeline(
        "question-answering",
        model=checkpoint,
        tokenizer=checkpoint,
        device=0 if torch.cuda.is_available() else -1,
    )
    rows=[]
    for ex in raw_datasets["test"]:
        t0=time.perf_counter()
        out=qa(question=ex["question"],context=ex["context"])
        rows.append({
            "id":ex["id"],
            "prediction":out["answer"],
            "gold":ex["answers"]["text"][0],
            "confidence":float(out["score"]),
            "latency_seconds":time.perf_counter()-t0,
        })
    return rows

checkpoints=sorted(
    glob.glob(f"{OUTPUT_DIR}/checkpoint-*"),
    key=lambda x:int(x.rsplit("-",1)[-1]),
)

epoch_rows=[]
for epoch,checkpoint in enumerate(checkpoints,1):
    predictions=evaluate_checkpoint(checkpoint)
    metrics=score_rows(predictions)
    loss_values=[
        float(x["loss"]) for x in trainer.state.log_history
        if "loss" in x and int(round(float(x.get("epoch",0))))==epoch
    ]
    metrics["epoch"]=epoch
    metrics["training_loss"]=loss_values[-1] if loss_values else None
    epoch_rows.append(metrics)

    with open(f"{OUTPUT_DIR}/test_predictions_epoch_{epoch}.jsonl","w",encoding="utf-8") as f:
        for row in predictions:
            f.write(json.dumps(row,ensure_ascii=False)+"\n")

epoch_metrics=pd.DataFrame(epoch_rows)
epoch_metrics.to_csv(f"{OUTPUT_DIR}/epoch_metrics.csv",index=False)
display(epoch_metrics)
